In [ ]:
!pip install pandas
import pandas as pd 
import sqlite3

* 1. Create a connection to the database using the sqlite3 library.

In [ ]:
conn = sqlite3.connect('../ex00/data/checking-logs.sqlite')

* 2. Get the schema of the test table.

In [ ]:
pd.io.sql.read_sql("PRAGMA table_info(test)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


* 3. Get only the first ten rows of the test table to see what it looks like.

In [ ]:
first_ten = pd.read_sql_query("SELECT * FROM test LIMIT 10", conn)
first_ten

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


* 4. Find the minimum value of the delta between the first commit and the deadline of the corresponding lab for all users using only one query.

    - Do this by joining the table with the deadlines table.
    - The difference should be displayed in hours.
    - Do not take lab project1 into account; it has longer deadlines and will be an outlier.
    - The value should be stored in the dataframe df_min with the corresponding uid.

In [ ]:
delta_min = """
    SELECT 
        t.uid,
        MIN(ROUND((JULIANDAY(t.first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24, 2)) AS min_diff_hours
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid
    ORDER BY min_diff_hours ASC
    LIMIT 1
"""

df_min = pd.read_sql_query(delta_min, conn)
df_min

,uid,min_diff_hours
0,user_30,-202.38


* 5. Do the same thing for the maximum, but use only one query. The dataframe name is df_max.

In [ ]:
delta_max = """
    SELECT 
        t.uid,
        MAX(ROUND((JULIANDAY(t.first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24, 2)) AS max_diff_hours
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
    GROUP BY t.uid
    ORDER BY max_diff_hours ASC
    LIMIT 1
"""

df_max = pd.read_sql_query(delta_max, conn)
df_max

,uid,max_diff_hours
0,user_14,-84.45


* 6. Do the same thing, but for the average. Use only one query. This time, your dataframe should not include the uid column. The dataframe name is df_avg.

In [ ]:
delta_avg = """
    SELECT 
        ROUND(AVG((JULIANDAY(t.first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24), 2) AS avg_diff_hours
    FROM test t
    JOIN deadlines d ON t.labname = d.labs
    WHERE t.labname != 'project1'
"""

df_avg = pd.read_sql_query(delta_avg, conn)
df_avg

,avg_diff_hours
0,-89.69


* 7. We want to test the hypothesis that users who visited the newsfeed just a few times have a lower delta between the first commit and the deadline. To do this, calculate the correlation coefficient between the number of pageviews and the difference.

  Using only one query, create a table with the following columns: "uid", "avg_diff", and "pageviews".
    - "uid" is the uids that exist in the test.
    - "avg_diff" is the average delta between the first commit and the lab deadline per user.
    - "pageviews" is the number of Newsfeed visits per user.
    - Do not take the lab project1 into account.
    - Store it in the dataframe views_diff.

In [ ]:
new_table = """
    SELECT
        t.uid,
        ROUND(AVG((JULIANDAY(t.first_commit_ts) - JULIANDAY(datetime(d.deadlines, 'unixepoch'))) * 24), 2) AS avg_diff,
        COUNT(p.datetime) AS pageviews
    FROM test t
    JOIN deadlines d ON t.labname == d.labs
    LEFT JOIN pageviews p ON t.uid = p.uid
    WHERE labname != 'project1'
    GROUP BY t.uid
"""

views_diff = pd.read_sql_query(new_table, conn)
views_diff

,uid,avg_diff,pageviews
0,user_1,-65.12,140
1,user_10,-75.24,445
2,user_14,-159.57,429
3,user_17,-62.21,235
4,user_18,-6.37,9
5,user_19,-99.44,64
6,user_21,-96.11,40
7,user_25,-93.47,895
8,user_28,-86.79,745
9,user_3,-105.74,1585


* Use the Pandas corr() method to calculate the correlation coefficient between the number of pageviews and the difference.

In [ ]:
print(views_diff[['avg_diff', 'pageviews']].corr())

           avg_diff  pageviews
avg_diff   1.000000  -0.185029
pageviews -0.185029   1.000000


* 8. Close the connection.

In [ ]:
conn.close()